# BayesianLoRA - Deferral Mechanism Testing (Colab)


## Step 1: Setup Environment

In [ ]:
# Clon  repo 
!git clone https://github.com/JohnD26/bayesian-peft.git

%cd bayesian-peft

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Login to WandB (optional but recommended)
import wandb
wandb.login()
# Then login to huggingface as we have a gated repo which is being used

## Step 2: Quick Test - Backward Compatibility 

In [ ]:
# Test 1: Verify existing code still works (Without deferral)
!bash scripts/custom/test-backward-compatibility.sh

## Step 3:  Deferral Mechanism

### Option A:  Deferral Test 

In [ ]:
# Quick test with deferral enabled on USMLE only
!python run/main.py \
  --dataset-type mcdataset --dataset usmle \
  --model-type causallm --model ContactDoctor/Bio-Medical-Llama-3-8B \
  --modelwrapper blob \
  --load-in-8bit False \
  --lr 1e-4 --batch-size 4 \
  --opt adamw --warmup-ratio 0.06 \
  --max-seq-len 512 \
  --seed 1 \
  --evaluate \
  --wandb-entity jonathandomingue15-university-of-ottawa \
  --wandb-project BLoB-deferral-quick-test \
  --wandb-name quick-deferral-test \
  --log-path quick-test \
  --max-train-steps 50 \
  --eval-per-steps 25 \
  --bayes-klreweighting \
  --bayes-eps 0.05 --bayes-beta 0.2 --bayes-gamma 8 --bayes-kllr 0.01 \
  --bayes-datasetrescaling \
  --bayes-train-n-samples 1 --bayes-eval-n-samples 1 --bayes-eval-n-samples-final 10 \
  --apply-classhead-lora --lora-r 8 --lora-alpha 16 --lora-dropout 0 \
  --enable-deferral \
  --deferral-threshold 0.12 \
  --deferral-metric max_std \
  --deferral-strategy exclude \
  --deferral-log-to-wandb

Look for the **Deferral Summary** in the output above!

### Option B: Run Individual Test Scripts

In [ ]:
# Test with "exclude" strategy (deployment mode)
!bash scripts/custom/test-deferral-exclude.sh

In [ ]:
# Test with "always_predict" strategy (research mode)
!bash scripts/custom/test-deferral-always-predict.sh

In [ ]:
# Test extreme thresholds (verification)
!bash scripts/custom/test-deferral-extreme-thresholds.sh

In [ ]:
# Compare uncertainty metrics
!bash scripts/custom/test-deferral-uncertainty-metrics.sh

### Option C: Run Full Test Suite (2-4 hours)

In [ ]:
# WARNING: This takes 2-4 hours!
!bash scripts/custom/test-deferral-full-suite.sh

## Step 4: View Results

### Check JSONL Log Files

In [ ]:
# Find all deferred sample logs
!find checkpoints -name "deferred_samples.jsonl" -type f

In [ ]:
# View a deferred samples file (replace with your actual path)
import json

log_file = "checkpoints/blob/ContactDoctor/Bio-Medical-Llama-3-8B/usmle/deferred_samples.jsonl"

with open(log_file, 'r') as f:
    for i, line in enumerate(f):
        if i >= 5:  # Show first 5 deferred samples
            break
        sample = json.loads(line)
        print(f"\n=== Deferred Sample {i+1} ===")
        print(f"Question: {sample['question'][:100]}...")
        print(f"Predicted: {sample['predicted_answer']} (uncertainty: {sample['uncertainty_score']:.4f})")
        print(f"True answer: {sample['true_answer']}")
        print(f"Correct: {sample['is_correct']}")

In [ ]:
# Count total deferred samples
!wc -l checkpoints/blob/*/usmle/deferred_samples.jsonl

### Analyze Results

In [ ]:
# Load and analyze deferred samples
import pandas as pd
import matplotlib.pyplot as plt

# Load JSONL into DataFrame
samples = []
with open(log_file, 'r') as f:
    for line in f:
        samples.append(json.loads(line))

df = pd.DataFrame(samples)

print(f"Total deferred: {len(df)}")
print(f"Incorrectly predicted: {(~df['is_correct']).sum()}")
print(f"Mean uncertainty: {df['uncertainty_score'].mean():.4f}")
print(f"Max uncertainty: {df['uncertainty_score'].max():.4f}")

# Plot uncertainty distribution
plt.figure(figsize=(10, 4))
plt.hist(df['uncertainty_score'], bins=30, edgecolor='black')
plt.xlabel('Uncertainty Score')
plt.ylabel('Count')
plt.title('Distribution of Uncertainty Scores (Deferred Samples)')
plt.show()

# Show most uncertain questions
print("\n=== Top 3 Most Uncertain Questions ===")
for idx, row in df.nlargest(3, 'uncertainty_score').iterrows():
    print(f"\nUncertainty: {row['uncertainty_score']:.4f}")
    print(f"Question: {row['question'][:150]}...")
    print(f"Predicted: {row['predicted_answer']}, True: {row['true_answer']}")

### Download Results 

In [ ]:
# Download JSONL file to your local machine
from google.colab import files

files.download(log_file)

In [ ]:
# Or save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

!cp -r checkpoints /content/drive/MyDrive/bayesian-peft-results/

## Step 5: View WandB Dashboard

Open: https://wandb.ai/jonathandomingue15-university-of-ottawa/BLoB-deferral-tests

